INGESTA DE DATOS

In [21]:
import os
import json
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

from pydantic import BaseModel, Field
from typing import List, Optional

load_dotenv()

True

In [22]:
# Only for tests, better results without schema
class ActuatorData(BaseModel):
    base_part_number: str = Field(description="The unique identifier for the actuator model, e.g., 761A00-11300000/A")
    enclosure_type: str = Field(description="Type of enclosure: 'Weatherproof' or 'Explosionproof'")
    voltage: str = Field(description="Operating voltage, e.g., '110V', '220V', '24V'")
    phase: str = Field(description="Power phase, e.g., 'Single Phase', '3 Phase', 'AC/DC'")
    
    on_off_output_torque_nm: Optional[float] = Field(None, description="On/Off Output Torque in Nm")
    on_off_output_torque_in_lbs: Optional[float] = Field(None, description="On/Off Output Torque in In-Lbs")
    on_off_duty_cycle_percent: Optional[int] = Field(None, description="On/Off Duty Cycle S4 percentage")
    cycles_per_hour: Optional[int] = Field(None, description="Allowable cycles per hour for On/Off")
    
    modulating_output_torque_nm: Optional[float] = Field(None, description="Modulating Output Torque in Nm")
    modulating_output_torque_in_lbs: Optional[float] = Field(None, description="Modulating Output Torque in In-Lbs")
    modulating_duty_cycle_percent: Optional[int] = Field(None, description="Modulating Duty Cycle S4 percentage")
    starts_per_hour: Optional[int] = Field(None, description="Allowable starts per hour for Modulating")
    
    operating_speed_60hz_sec: Optional[float] = Field(None, description="Operating speed in seconds for 60 Hz")
    operating_speed_50hz_sec: Optional[float] = Field(None, description="Operating speed in seconds for 50 Hz")
    full_load_current_fla_60hz_amps: Optional[float] = Field(None, description="Full Load Current (FLA) in Amps for 60 Hz")
    full_load_current_fla_50hz_amps: Optional[float] = Field(None, description="Full Load Current (FLA) in Amps for 50 Hz")
    locked_rotor_current_lra_60hz_amps: Optional[float] = Field(None, description="Locked Rotor Current (LRA) in Amps for 60 Hz")
    locked_rotor_current_lra_50hz_amps: Optional[float] = Field(None, description="Locked Rotor Current (LRA) in Amps for 50 Hz")
    motor_power_watts: Optional[int] = Field(None, description="Motor power rating in Watts")

class ActuatorDataSet(BaseModel):
    actuators: List[ActuatorData] = Field(description="List of all extracted actuators from the document tables without omissions")

In [23]:
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

In [24]:
pdf_path = "D:\\Development\\intelligent-actuator-data-agent\\data\\raw\\series_76_tables.pdf"

print("Uploading PDF to Google GenAI...")
file_ref = client.files.upload(file= pdf_path)
print("file uploaded successfully")

Uploading PDF to Google GenAI...
file uploaded successfully


In [28]:
system_prompt = """
Act as a data extraction expert. Analyze the attached PDF for the Series 76 actuators. 
Extract every single row from the technical tables and convert them into a single structured list in JSON format. 
It is essential that you associate each row with its page metadata: identify the Enclosure Type 
(Weatherproof or Explosionproof), the Voltage (110V, 220V, 24V), and the Phase (Single or 3-Phase) 
based on the headings of the page where the row is located. 
Explicitly separate the 50 Hz and 60 Hz values into independent numerical columns where applicable. 
Do not omit any model.
"""

response = client.models.generate_content(
    model= "gemini-2.5-flash",
    contents = [file_ref, "Extract the data from this PDF according to the system instructions."],
    config = types.GenerateContentConfig(
        system_instruction = system_prompt,
        response_mime_type = "application/json",
        #response_schema = ActuatorDataSet,
        temperature = 0.0,
    )
)

print (response.text)

[
  {
    "Enclosure Type": "Weatherproof",
    "Voltage": "110V",
    "Phase": "Single",
    "Base Part Number": "761A00-11300000/A",
    "On/Off Output Torque In-Lbs": 700,
    "On/Off Output Torque Nm": 80,
    "On/Off Cycles Duty Per Cycle Hour S4%": 40,
    "On/Off Cycles Duty Per Cycle Hour Cycles": 36,
    "Modulating Output Torque In-Lbs": 580,
    "Modulating Output Torque Nm": 65,
    "Modulating Duty Cycle S4%": 40,
    "Starts Per Hour": 960,
    "Operating Speed (sec) 60 Hz": 17,
    "Operating Speed (sec) 50 Hz": 20,
    "Operating Speed (sec)": null,
    "Full Load Current (FLA) 60 Hz": 1.2,
    "Full Load Current (FLA) 50 Hz": 1.1,
    "Full Load Current (FLA) Amps": null,
    "Locked Rotor Current (LRA) 60 Hz": 1.4,
    "Locked Rotor Current (LRA) 50 Hz": 1.3,
    "Locked Rotor Current (LRA) Amps": null,
    "Motor Power Watts": 15
  },
  {
    "Enclosure Type": "Weatherproof",
    "Voltage": "110V",
    "Phase": "Single",
    "Base Part Number": "761B00-11300000/A",
 

In [29]:
json_data = json.loads(response.text)
df = pd.DataFrame(json_data)
df = df.drop(columns=['Operating Speed (sec)', 'Full Load Current (FLA) Amps', 'Locked Rotor Current (LRA) Amps'])
df["Motor Power Watts"] = df["Motor Power Watts"].bfill().ffill()

In [30]:
conn = sqlite3.connect("D:/Development/intelligent-actuator-data-agent/data/processed/electric_data_table.db")
df.to_sql("electric_data_table", conn, if_exists="replace", index=False)
conn.close()